## 1 — ライブラリの読込み

- 処理: ライブラリの読込み
- 入力: プロジェクトのPython環境
- 出力: 検証・集計・描画用の関数
- 次: 実験JSONを指定

- 表はクリックで展開 / 縦横スクロール / CSVは同じ列順の全行・全精度

- 共通評価は新規発見・継続的な嗜好・多様性／発見性・効率性 / 集計は02、定義は[評価仕様](../docs/evaluation.md)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from src.infrastructure.datasets.listening_data import validate_bundle
from src.interfaces.notebook.evaluation_figures import observation_figures, show_figures
from src.interfaces.notebook.listening_analysis import observation_distribution
from src.interfaces.notebook.notebook_config import ExperimentConfigPicker
from src.interfaces.notebook.notebook_display import show_table
from src.interfaces.notebook.search_analysis import validation_overview
from src.interfaces.notebook.split_figures import load_split_frame, split_figure

## 2 — 実験設定を選択

- 処理: configs内のlistening-v2設定を選択し、読み込むボタンで確定
- 入力: 選択欄のJSON / 初期状態は未選択
- 出力: 読み込んだ実験名・モデル・入力パス / 評価は開始しない
- 次: 読込み済みの表示を確認して次セルへ / ファイル追加後は一覧を更新


In [ ]:
ROOT = Path.cwd() if Path("src").exists() else Path.cwd().parent
config_picker = ExperimentConfigPicker(ROOT / "configs")
display(config_picker.widget)

## 3 — 選択した設定の確認

- 処理: 明示的に読み込んだ設定を後続セルで使用
- 入力: 選択欄の読込み済み設定
- 出力: WORKFLOWと実験名 / 未指定なら案内して停止
- 次: 後続セルを上から順に実行 / 別のNotebookでも同じファイルを選択


In [ ]:
WORKFLOW = config_picker.require_loaded()
EXPERIMENT_CONFIG = WORKFLOW.path
print("実験:", WORKFLOW.name)

## 4 — 保存済み分割の読込み

- 処理: 選択したValidationと同じ出典を照合し、2002年より前の再生を除外した表示用集計を作成
- 入力: 同じ入力一式のready.json・source/summary.json・Validation manifest
- 出力: Train / Validation / Testのイベント数・割合・ユニーク楽曲数・期間 / 元の入力・評価結果は保持
- 次: 英語の分割比率グラフを表示 / 評価は実行しない


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
try:
    split_df = load_split_frame(WORKFLOW.validation_dataset, exclude_before="2002-01-01T00:00:00")
except FileNotFoundError:
    split_df = None
    print("分割集計が未準備 / 選択した入力一式のready.jsonとsource/summary.jsonを確認してください")

## 5 — 分割比率の表示

- 処理: イベント数に比例する積み上げ棒グラフをMatplotlibで描画
- 入力: 読込み済みの分割集計 / 横軸はイベント数の割合 / tracksは期間ごとのユニーク楽曲数で期間間の重複あり
- 出力: 両端と境界の日付・割合・楽曲数を示す英語図・300 dpi PNG・詳細表とCSV / artifacts/<dataset_id>/split.pngへ保存
- 次: Validation入力を検証して候補・正解と再生分布を確認


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
if split_df is None:
    print("分割集計が未準備 / 前のセルで入力一式を確認してください")
else:
    split_fig = split_figure(split_df)
    split_png = ROOT / "artifacts" / split_df.attrs["dataset_id"] / "split.png"
    split_png.parent.mkdir(parents=True, exist_ok=True)
    split_fig.savefig(split_png, dpi=300, facecolor="white")
    display(split_fig)
    plt.close(split_fig)
    print("PNG:", split_png)
    print("表示集計から除外した再生記録:", split_df.attrs["excluded_events"])
    show_table(
        split_df[["partition", "share_pct", "events", "tracks", "start", "end"]],
        "分割比率と期間",
        "01-split-ratios",
    )

## 6 — Validation入力の検証

- 処理: Validation入力の検証
- 入力: JSONのvalidation_dataset
- 出力: hash・型・観測整合性を照合したmanifestと学習件数
- 次: 学習母集団と分割境界を確認


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
if not (WORKFLOW.validation_dataset / "manifest.json").exists():
    raise FileNotFoundError(
        "Validation入力が未準備 / 00_lab_startで取得・変換ログを確認してください"
    )
manifest, paths, counts = validate_bundle(WORKFLOW.validation_dataset)
print("入力照合済み:", manifest["dataset_id"])
show_table(pd.DataFrame([counts]), "学習データ件数", "01-training-counts")
show_table(
    pd.DataFrame(manifest["files"]).T.rename_axis("file").reset_index(),
    "入力ファイル",
    "01-input-files",
)

## 7 — 分割条件と評価可能性の確認

- 処理: 分割条件と評価可能性の確認
- 入力: 同じValidation入力とJSONの評価条件
- 出力: 境界・窓・new/repeat別の候補数と正解保有ユーザー数
- 次: 観測の分布を確認


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
evaluation_conditions, validation_requests_df, overview = validation_overview(WORKFLOW)
print("予測時点:", manifest["prediction_at"], "/ 評価窓（日）:", manifest["window_days"])
request_summary = (
    validation_requests_df.groupby("task", sort=False)
    .agg(
        users=("user_id", "nunique"),
        evaluable_users=("positive_count", lambda values: (values > 0).sum()),
        mean_candidates=("candidate_count", "mean"),
        mean_positives=("positive_count", "mean"),
    )
    .reset_index()
)
request_summary.set_index("task")[["users", "evaluable_users"]].rename(
    columns={"users": "All users", "evaluable_users": "With positives"}
).plot.bar(rot=0, figsize=(6, 3), ylabel="Users", xlabel="Task", title="Evaluation eligibility")
plt.tight_layout()
plt.show()
show_table(request_summary, "評価可能性", "01-eligibility")
show_table(pd.DataFrame([manifest["temporal"]]), "分割条件", "01-split")
show_table(
    pd.Series(overview, name="値").rename_axis("項目").reset_index(), "評価窓の詳細", "01-windows"
)

## 8 — 再生回数・再生日数の分布

- 処理: 再生回数・再生日数の分布
- 入力: 主評価窓のValidation観測
- 出力: new/repeat別のヒストグラムと集計表
- 次: 00_lab_startでSmoke / Search → 02_analysisで分析


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
observation_df = observation_distribution(WORKFLOW)
if observation_df.empty:
    print("主評価窓に評価対象の再生観測がありません / 入力と期間を確認してください")
else:
    show_figures(observation_figures(observation_df))
show_table(
    observation_df.reindex(columns=["task", "observation", "bin", "pairs", "window_days"]),
    "観測分布の値",
    "01-observations",
)